In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


# =========================================================
# Paths
# =========================================================

CACHE_DIR = Path(
    # "/vol/biomedic3/tx1215/mamo-flow/cache/flux2_vae_512x384",
    "/vol/biomedic3/tx1215/mamo-flow/cache/flux2_vae_1024x768",

)

OUT_PATH = (
    CACHE_DIR
    / "pair_csv_latent.csv"
)



In [2]:

# =========================================================
# 1. Load latent-cache manifests
#
# manifest.iloc[i] <-> latent memmap[i]
# =========================================================

dfs = []

for split in ["train", "valid", "test"]:

    manifest_path = (
        CACHE_DIR
        / f"{split}_manifest.csv"
    )

    df_split = pd.read_csv(
        manifest_path,
        low_memory=False,
    ).reset_index(drop=True)

    df_split["split"] = split

    # -----------------------------------------------------
    # Strong consistency check:
    #
    # row i must correspond to latent cache i
    # -----------------------------------------------------

    cache_idx = (
        df_split["cache_idx"]
        .to_numpy(dtype=np.int64)
    )

    expected = np.arange(
        len(df_split),
        dtype=np.int64,
    )

    if not np.array_equal(
        cache_idx,
        expected,
    ):
        raise ValueError(
            f"{split}: manifest/cache mismatch. "
            "Expected cache_idx == manifest row index."
        )

    dfs.append(
        df_split
    )


df = pd.concat(
    dfs,
    ignore_index=True,
)

print(
    "Total cached latent samples:",
    len(df),
)

print(
    df["split"].value_counts()
)



Total cached latent samples: 297373
split
train    237746
test      36934
valid     22693
Name: count, dtype: int64


In [3]:

# =========================================================
# 2. Normalize view / laterality
# =========================================================

df["ViewPosition_norm"] = (
    df["ViewPosition"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df["Laterality_norm"] = (
    df["ImageLateralityFinal"]
    .astype(str)
    .str.upper()
    .str.strip()
)

In [4]:

# =========================================================
# 3. Keep standard 2D CC / MLO only
# =========================================================

if "cview" in df.columns:

    df = df[
        df["cview"] == 0
    ].copy()


if "spot_mag" in df.columns:

    spot_mag = (
        df["spot_mag"]
        .fillna(0)
        .astype(str)
        .str.lower()
    )

    df = df[
        spot_mag.isin(
            [
                "0",
                "0.0",
                "false",
            ]
        )
    ].copy()


df_std = df[
    df["Laterality_norm"].isin(
        ["L", "R"]
    )
    &
    df["ViewPosition_norm"].isin(
        ["CC", "MLO"]
    )
].copy()


print(
    "Standard cached CC/MLO latents:",
    len(df_std),
)



Standard cached CC/MLO latents: 297373


In [5]:

# =========================================================
# 4. Build same-patient / same-exam / same-breast pairs
# =========================================================

group_cols = [
    "split",
    "empi_anon",
    "acc_anon",
    "Laterality_norm",
]


pairs = []

skipped_missing = 0
skipped_ambiguous = 0


for (split, empi, acc, lat), g in df_std.groupby(
    group_cols,
    sort=False,
):

    cc_rows = g[
        g["ViewPosition_norm"] == "CC"
    ]

    mlo_rows = g[
        g["ViewPosition_norm"] == "MLO"
    ]


    # -----------------------------------------------------
    # Need both CC and MLO
    # -----------------------------------------------------

    if (
        len(cc_rows) == 0
        or len(mlo_rows) == 0
    ):
        skipped_missing += 1
        continue


    # -----------------------------------------------------
    # Keep clean one-to-one pairs for GT comparison
    # -----------------------------------------------------

    if (
        len(cc_rows) != 1
        or len(mlo_rows) != 1
    ):
        skipped_ambiguous += 1
        continue


    cc = cc_rows.iloc[0]
    mlo = mlo_rows.iloc[0]


    # =====================================================
    # IMPORTANT:
    #
    # These cache_idx values point DIRECTLY into:
    #
    # flux2encoding_float32_<split>.dat
    #
    # i.e.
    #
    # latent_cache[cc_cache_idx]
    # latent_cache[mlo_cache_idx]
    # =====================================================

    pair = {

        "split": split,

        "empi_anon": empi,
        "acc_anon": acc,
        "laterality": lat,

        "cc_cache_idx": int(
            cc["cache_idx"]
        ),

        "mlo_cache_idx": int(
            mlo["cache_idx"]
        ),

        "cc_path": cc["image_path"],
        "mlo_path": mlo["image_path"],
    }


    # -----------------------------------------------------
    # Keep all metadata for later analysis/debugging
    # -----------------------------------------------------

    for col in df.columns:

        pair[f"cc_{col}"] = cc[col]
        pair[f"mlo_{col}"] = mlo[col]


    pairs.append(
        pair
    )



In [ ]:

# =========================================================
# 5. Save
# =========================================================

pairs_df = pd.DataFrame(
    pairs
)

pairs_df.to_csv(
    OUT_PATH,
    index=False,
)


# =========================================================
# 6. Summary
# =========================================================

print()

print(
    "Number of valid latent CC/MLO pairs:",
    len(pairs_df),
)

print(
    "Skipped - missing CC or MLO:",
    skipped_missing,
)

print(
    "Skipped - ambiguous multiple views:",
    skipped_ambiguous,
)

print()

print(
    "Pairs per split:"
)

print(
    pairs_df["split"]
    .value_counts()
)

print()

print(
    "Saved to:",
    OUT_PATH,
)


pairs_df.head()

